# 06 · Experiments against the frozen reference B0

| id | change vs B0 | motivation |
|---|---|---|
| X1 | one model per language, each with its own feature/regularisation search | EN and DE differ in domains, length behaviour, casing and compounding (`01_eda`) |
| X2 | two-stage: stage-1 domain classifier → per-domain expert | class priors and AI style differ by domain |
| X2g | X2 with routing by the given `domain` column | isolates the cost of stage-1 routing errors |
| X3 | LM likelihood + segment features → gradient boosting | model-based signal independent of n-grams; segment spread aimed at fusion |
| E1 | fine-tuned multilingual encoder (from `04_encoder`) | required encoder comparison |

Every run writes `artifacts/runs/<id>.json` (config, compute, runtime, status, error if it failed) and predictions to `artifacts/preds/`. The log at the end is rebuilt from these files.

In [ ]:
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning

from hw3.config import ROOT, art, load_config

warnings.filterwarnings("ignore", category=ConvergenceWarning)
pd.set_option("display.width", 200, "display.max_columns", 40)
FIG = ROOT / "images"

from hw3 import lm, tfidf
from hw3.runs import compare, experiment_log, has_preds, report

## X1 · language-specific models

In [ ]:
grids = tfidf.run_lang_tuned(load_config("lang_tuned"))
pd.concat(grids, names=["lang"])[["features", "model", "macro_f1", "f1_human", "f1_ai", "f1_fusion"]]

In [ ]:
compare(["B0", "X1"], by=["lang"])

## X2 · two-stage domain routing

In [ ]:
routes = tfidf.run_two_stage(load_config("two_stage"))
meta = json.loads(art("runs", "X2.json").read_text())
meta["domain_accuracy_val"], meta["experts"]

In [ ]:
pd.crosstab(np.asarray(routes["val"]["given"]), np.asarray(routes["val"]["predicted"]), rownames=["given"], colnames=["predicted"])

In [ ]:
compare(["B0", "X2", "X2g"])

## X3 · LM likelihood features

Needs `artifacts/features/lm_*.parquet` from `05_lm_features`.

In [ ]:
if art("features", "lm_train.parquet").exists():
    lm.run_classifier(load_config("lm_features"))
compare(["B0", "X3"])

## E1 · encoder vs baseline

Needs `artifacts/preds/E1_*.parquet` from `04_encoder`.

In [ ]:
compare(["B0", "E1"])

## Per-class metrics of every model

In [ ]:
for model_id in ["B0", "X1", "X2", "X2g", "X3", "E1"]:
    if has_preds(model_id, "val"):
        rep = report(model_id)
        rep["figure"].savefig(FIG / f"confusion_{model_id}.png", dpi=150, bbox_inches="tight")
        display(rep["per_class"].rename_axis(model_id))

## Experiment log (`experiments/log.csv`)

In [ ]:
experiment_log()